# Code mẫu — Embedding và dữ liệu chuỗi

Token ID chỉ là số thứ tự của một hàng trong bảng embedding. Padding chỉ là phần đệm để các chuỗi dài ngắn khác nhau ghép được thành một bảng chữ nhật. Các ví dụ dưới đây tách rõ ba việc: đánh dấu token nào là thật, gộp các vector lại, và xem gradient đi tới đâu. Mục tiêu là thêm bao nhiêu PAD cũng không làm đổi ý nghĩa của câu.

## Ví dụ 1: Ghép chuỗi và tạo mask

Ba chuỗi token có độ dài khác nhau. Ta quy ước `PAD = 0` và `UNK = 1`, với UNK dành cho token không có trong từ điển. ID là số nguyên dùng để tra bảng. Token 4 không “lớn hơn” token 2 theo nghĩa nào cả.

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

torch.set_num_threads(1)
torch.manual_seed(381)
PAD, UNK = 0, 1
sequences = [
    torch.tensor([2, 3]),
    torch.tensor([UNK]),
    torch.tensor([4, 2, 3]),
]
ids = pad_sequence(sequences, batch_first=True, padding_value=PAD)
valid_mask = ids.ne(PAD)
lengths = valid_mask.sum(dim=1)
print("ids:", ids.tolist())
print("valid mask:", valid_mask.int().tolist())
print("lengths:", lengths.tolist())
print("shape:", tuple(ids.shape), "mask shape:", tuple(valid_mask.shape))
assert ids.dtype == torch.long
assert lengths.tolist() == [2, 1, 3]

ids: [[2, 3, 0], [1, 0, 0], [4, 2, 3]]
valid mask: [[1, 1, 0], [1, 0, 0], [1, 1, 1]]
lengths: [2, 1, 3]
shape: (3, 3) mask shape: (3, 3)


Sau khi đệm, ta có một bảng shape `(B, L) = (3, 3)`: ba chuỗi, mỗi chuỗi dài 3. Mask có cùng shape, đánh 1 cho token thật và 0 cho PAD. Đếm số 1 trên mỗi hàng là ra độ dài thật của chuỗi. Chuỗi thứ hai chỉ có một token UNK. UNK vẫn là token thật, nên vẫn được đánh 1. Chỉ PAD mới bị loại khỏi các phép gộp phía sau.

## Ví dụ 2: Lấy trung bình có mask, và kiểm khi thêm PAD

Ta đặt một bảng embedding nhỏ bằng tay để tính được từng tọa độ. Hàng PAD toàn số 0. Nhưng như thế chưa đủ. Vector 0 chỉ giữ tổng không đổi, còn khi chia, mẫu số vẫn đếm cả PAD. Vì vậy mask phải có mặt ở cả tử số lẫn mẫu số.

In [ ]:
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence

torch.set_num_threads(1)
torch.manual_seed(382)
PAD = 0
sequences = [torch.tensor([2, 3]), torch.tensor([3]), torch.tensor([4, 2, 3])]
ids = pad_sequence(sequences, batch_first=True, padding_value=PAD)
valid_mask = ids.ne(PAD)
table = torch.tensor([
    [0., 0.],  # PAD
    [1., 1.],  # UNK
    [2., 0.],  # token 2
    [0., 4.],  # token 3
    [2., 2.],  # token 4
])
embedding = nn.Embedding(5, 2, padding_idx=PAD)
with torch.no_grad():
    embedding.weight.copy_(table)

def masked_mean(vectors, mask):
    assert vectors.shape[:2] == mask.shape
    counts = mask.sum(dim=1, keepdim=True)
    if torch.any(counts == 0):
        raise ValueError("empty sequence has no valid token")
    return (vectors * mask.unsqueeze(-1)).sum(dim=1) / counts

vectors = embedding(ids)
pooled = masked_mean(vectors, valid_mask)
naive = vectors.mean(dim=1)
print("embedding shape:", tuple(vectors.shape))
print("masked mean:", [[round(v, 4) for v in row] for row in pooled.tolist()])
print("naive first mean:", [round(v, 4) for v in naive[0].tolist()])

extra_ids = torch.cat([ids, torch.zeros(len(ids), 2, dtype=torch.long)], dim=1)
extra_pooled = masked_mean(embedding(extra_ids), extra_ids.ne(PAD))
print("adding PAD changes nothing:", torch.allclose(pooled, extra_pooled))
try:
    empty = torch.zeros(1, 3, dtype=torch.long)
    masked_mean(embedding(empty), empty.ne(PAD))
except ValueError as error:
    print("empty sequence rejected:", str(error))
assert torch.allclose(pooled[0], torch.tensor([1., 2.]))
assert not torch.allclose(pooled[0], naive[0])
assert torch.allclose(pooled, extra_pooled)

embedding shape: (3, 3, 2)
masked mean: [[1.0, 2.0], [0.0, 4.0], [1.3333, 2.0]]
naive first mean: [0.6667, 1.3333]
adding PAD changes nothing: True
empty sequence rejected: empty sequence has no valid token


Sau khi tra bảng, shape là `(3, 3, 2)`: mỗi token biến thành một vector hai chiều. Câu đầu có hai token thật, vector `[2, 0]` và `[0, 4]`, nên trung bình đúng là `[1, 2]`. Lấy trung bình thường thì chia cho 3, tính cả ô PAD, và vector bị co lại thành `[0.6667, 1.3333]`. Nghĩa của câu bị kéo theo số ô đệm.

Thêm hai cột PAD nữa vào mọi câu, kết quả có mask vẫn y nguyên. Còn một câu không có token thật nào thì bị từ chối, kèm một thông báo rõ ràng, thay vì lặng lẽ chia cho 0 rồi trả về NaN.

## Ví dụ 3: Hàng nào của embedding nhận gradient, và thứ tự bị mất ra sao?

Khi tra bảng, gradient chỉ đi về những hàng ứng với token có mặt trong batch. Ta kiểm điều đó, rồi xem một hạn chế của phép lấy trung bình: đổi thứ tự token thì kết quả không đổi. Như vậy, model chỉ dùng trung bình sẽ không phân biệt được “A rồi B” với “B rồi A”.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(383)
PAD = 0
embedding = nn.Embedding(5, 2, padding_idx=PAD)
with torch.no_grad():
    embedding.weight.copy_(torch.tensor([
        [0., 0.], [1., 1.], [2., 0.], [0., 4.], [2., 2.]
    ]))
ids = torch.tensor([[2, 3, 0], [3, 0, 0], [4, 2, 3]])
embedding(ids).sum().backward()
grad = embedding.weight.grad
print("gradient row 0 (PAD):", grad[PAD].tolist())
print("gradient rows 2,3,4:", grad[2:5].tolist())
print("token counts 2,3,4:", [2, 3, 1])

def mean_without_pad(token_ids):
    vectors = embedding(token_ids)
    mask = token_ids.ne(PAD)
    return (vectors * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True)

pair = torch.tensor([[2, 3], [3, 2]])
pair_pooled = mean_without_pad(pair)
print("swapping token order keeps mean:", torch.allclose(pair_pooled[0], pair_pooled[1]))
print("pair means:", pair_pooled.tolist())
assert grad[PAD].eq(0).all()
assert torch.allclose(grad[2], torch.tensor([2., 2.]))
assert torch.allclose(grad[3], torch.tensor([3., 3.]))
assert torch.allclose(pair_pooled[0], pair_pooled[1])

gradient row 0 (PAD): [0.0, 0.0]
gradient rows 2,3,4: [[2.0, 2.0], [3.0, 3.0], [1.0, 1.0]]
token counts 2,3,4: [2, 3, 1]
swapping token order keeps mean: True
pair means: [[1.0, 2.0], [1.0, 2.0]]


Loss ở đây là tổng mọi tọa độ, nên mỗi lần một token xuất hiện, hàng của nó nhận thêm 1 ở mỗi tọa độ. Token 2 xuất hiện hai lần, nên gradient hàng 2 là `[2, 2]`. Token 3 xuất hiện ba lần, nên là `[3, 3]`. Hàng UNK không xuất hiện nên không nhận gì. Hàng PAD xuất hiện bốn lần mà gradient vẫn bằng 0, nhờ `padding_idx`. Vector PAD vì thế đứng yên trong suốt quá trình train.

Câu `[2, 3]` và câu `[3, 2]` cho cùng một trung bình, vì phép cộng không quan tâm thứ tự. Với những bài mà thứ tự từ mang nghĩa, model cần thêm thông tin về vị trí của từng token. Các bài sau sẽ bàn cách làm việc này.